In [0]:
df = spark.read.table('development.bronze.tickets_bronze')

In [0]:
df.display()


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

In [0]:

columns = [
'created_at'

,'resolved_at'
]
df1 = df.withColumns({c+'_date': F.col(c).cast(DateType()) for c in columns})

In [0]:
df1.show(10)

In [0]:
columns = {
'created_at': 'created_at_date'

,'resolved_at': 'reolved_at_date'

}

In [0]:
from pyspark.sql import functions as F
rules_dict = {

    'valid_age' : 'age > 25',
    'non_francophone' : 'province <> "QC" '
}

rule_checks =[ 
                           F.when(~F.expr(cond) | F.expr(cond).isNull(), F.lit(rule_name)).otherwise(None) 
                           for rule_name, cond in rules_dict.items()
            ]

In [0]:
rule_checks

In [0]:
def quarantine_data():
    """
    Apply quarantine logic to customer data.
    Returns a tuple of (clean_df, quarantine_df).
    """
    # Define quarantine logic (converted from DLT to standard notebook)
    rules_dict = {
        'valid_age' : 'age > 25',
        'non_francophone' : 'province <> "QC"'  # Fixed: removed trailing space
    }

    rule_checks = [ 
        F.when(~F.expr(cond) | F.expr(cond).isNull(), F.lit(rule_name)).otherwise(None) 
        for rule_name, cond in rules_dict.items()
    ]

    df = spark.read.table('development.bronze.cutomer_bronze')

    df_flagged = df.withColumns({
        'failed_rules': F.filter(F.array(*rule_checks), lambda x: x.isNotNull()),  # Fixed: use filter instead of array_remove
        'quarantined_at': F.current_timestamp()
    })

    df_quarantine = df_flagged.filter(F.size(F.col('failed_rules')) > 0)
    df_clean = df_flagged.filter(F.size(F.col('failed_rules')) == 0)

    return df_clean, df_quarantine

# Write to table
#df_quarantine.write.mode('overwrite').saveAsTable('development.bronze.test_quarantine')

#display(df_quarantine)



In [0]:
df_clean, df_quarantine = quarantine_data()
df_clean.show()
